[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/specular_caustics/specular_caustics.ipynb)

# Specular Caustics

Caustics are the bright curves where neighbouring rays of light crowd together. Sunlight through moving ripples on water draws a shifting network of them on the bottom: each ripple bends the rays that cross it, and where the bending folds the bundle over itself the light piles up.

Positions and ray directions are antivectors, and the wavevectors of the ripples and the plane of the water surface are covectors, vectors; a wave's phase at a point is their pairing, `wavevector & position`. The map from where a ray enters the water to where it lands is a map of horizontal positions, and its derivative, `derivative(landing)(chart)`, a map of horizontal displacements. Its determinant measures how a patch of incoming light spreads, vanishing on the folds, and its `solve` turns a miss at the receiver into a correction on the water.

The animation shows the caustics moving with the waves; a still marks the folds the determinant finds, and a side view shows three rays through one ripple meeting at one point.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import partial

import jax
import numpy as np

from numga.algebras import VGA3D as ga
from numga.backend.jax import JaxContext, derivative
from examples.animation import save_figure, show_animation
from examples.optics.specular_caustics import render

In [ ]:
jax.config.update("jax_enable_x64", True)
mv = JaxContext(ga, dtype=np.float64).multivector
Scalar = ga.gatype.scalar()
# Positions and directions are antivectors; wavevectors and planes are covectors, vectors.
Direction = ga.gatype.antivector()
Plane = ga.gatype.vector()
# Horizontal positions are the antivectors that contain z; wavevectors lie across the horizontal.
Horizontal = Direction.derive.containing("z")
Wavevector = Plane.derive.within("x y")
x, y = mv.x.dual(), mv.y.dual()
up = mv.z.dual()

## 1. Light through moving ripples

Parallel rays fall straight down onto water whose surface is a sum of travelling waves. The surface is the level set of its height above them, so its plane at a point is `mv.z - gradient`, and each ray bends across that plane by Snell's law. The receiver records power per area, with full transmission and a small pixel footprint, while the waves move as prescribed. This cell takes about ten seconds.

In [ ]:
wave_angles = np.array([0, 0.7, 1.5, 2.2])
wavenumbers = np.array([6, 8, 10, 12])
amplitudes = np.array([0.045, 0.035, 0.025, 0.018])
phases = np.array([0, 1.2, -0.7, 0.6])
phase_rates = np.array([1, -1, 2, -2])
index_ratio = 0.75
depth = 2.5
source_halfwidth = 1.6
receiver_halfwidth = 1
n_rays = 640
n_frames = 72
n_pixels = 440
blur_pixels = 1.4
exposure = 0.3
seed = 17
duration_ms = 70

wavevectors = ((-mv.xy * wave_angles / 2).exp() >> mv.x) * wavenumbers   # [waves] Wavevector
spacing = 2 * source_halfwidth / n_rays
samples = (np.arange(n_rays) + 0.5) * spacing - source_halfwidth
rng = np.random.default_rng(seed)
# One ray in each cell over the water, with a fixed jitter that hides the sampling lattice.
x_offsets = rng.uniform(-0.5, 0.5, (n_rays, n_rays)) * spacing
y_offsets = rng.uniform(-0.5, 0.5, (n_rays, n_rays)) * spacing
positions = x * (samples[None, :] + x_offsets) + y * (samples[:, None] + y_offsets)   # [rows, columns] Horizontal
ray_area = spacing**2

In [ ]:
def rays(
    position: Horizontal, wavevectors: Wavevector, amplitudes: np.ndarray,
    phases: np.ndarray, index_ratio: float,
) -> tuple[Direction, Plane, Direction]:
    """Surface positions, unit surface planes and transmitted directions for vertical illumination."""
    phase = (wavevectors & position[..., None]) - phases        # [..., waves] Scalar
    height = (amplitudes * phase.cos()).sum(axis=-1)            # [...] Scalar
    gradient = -(wavevectors * (amplitudes * phase.sin())).sum(axis=-1)   # [...] Wavevector
    # The plane of the surface, `z - height == 0`, and the unit direction across it.
    plane = (mv.z - gradient).normalized()                      # [...] Plane
    normal = plane.dual()                                       # [...] Direction

    # Snell's law scales the part of the ray along the surface; unit length fixes the part across it.
    incoming = -up                                              # Direction
    tangent = index_ratio * (incoming - normal * (plane & incoming))   # [...] Direction
    outgoing = tangent - normal * (1 - tangent.scalar_norm_squared()).square_root()   # [...] Direction
    surface = position + up * height                            # [...] Direction
    return surface, plane, outgoing


def receiver(
    position: Horizontal, wavevectors: Wavevector, amplitudes: np.ndarray,
    phases: np.ndarray, depth: float, index_ratio: float,
) -> Horizontal:
    """Horizontal landing positions on the receiver below the refracting surface."""
    surface, _, outgoing = rays(position, wavevectors, amplitudes, phases, index_ratio)
    # The distance along the ray down to the receiver, at height minus the depth.
    distance = (-depth - (mv.z & surface)) / (mv.z & outgoing)  # [...] Scalar
    landing = surface + distance * outgoing                     # [...] Direction
    return landing.cast(Horizontal)


def caustics() -> Iterator[Horizontal]:
    for time in np.linspace(0, 2 * np.pi, n_frames, endpoint=False):
        yield receiver(positions, wavevectors, amplitudes, phases + phase_rates * time, depth, index_ratio)


show_animation(render.animate(caustics(), receiver_halfwidth, ray_area, n_pixels, blur_pixels, exposure),
               "specular_caustics", duration_ms)

## 2. Where the ray map folds

Each wave travels its own way, so the bright folds split, meet and move across the receiver. A patch of water collapses onto a curve where the landing map loses a direction, and there its determinant vanishes. The coral curves are those collapsed patches carried to the receiver, at the first frame of the animation: they run along the bright folds.

In [ ]:
n_derivative = 192
chart_samples = np.linspace(-source_halfwidth, source_halfwidth, n_derivative)
chart = x * chart_samples[None, :] + y * chart_samples[:, None]   # [rows, columns] Horizontal

In [ ]:
landing = partial(receiver, wavevectors=wavevectors, amplitudes=amplitudes, phases=phases,
                  depth=depth, index_ratio=index_ratio)
local = derivative(landing)(chart)                               # [rows, columns] Horizontal <- Horizontal
area = local.det()                                               # [rows, columns] Scalar
figure = render.draw_critical(landing(positions), landing(chart), area, receiver_halfwidth,
                              ray_area, n_pixels, blur_pixels, exposure)
save_figure(figure, "specular_caustics_folds")
figure

## 3. Several paths to one target

A bright region receives light through several surface points at once. Under a single ripple the branches show in a side view: Newton's method corrects each trial point by solving the local ray map against its miss, and three starting points find three paths to the same point on the receiver.

In [ ]:
ripple_wavenumber = np.array([4])
ripple_amplitude = np.array([0.12])
ripple_phase = np.array([0])
seed_offsets = np.array([-0.3, 0, 0.3])
section_halfwidth = 0.8
incident_height = 0.6
n_section = 300
n_fan = 45
n_iterations = 3

In [ ]:
ripple_wavevector = mv.x * ripple_wavenumber                     # [waves] Wavevector
ripple = partial(rays, wavevectors=ripple_wavevector, amplitudes=ripple_amplitude, phases=ripple_phase,
                 index_ratio=index_ratio)
ripple_landing = partial(receiver, wavevectors=ripple_wavevector, amplitudes=ripple_amplitude,
                         phases=ripple_phase, depth=depth, index_ratio=index_ratio)
target = 0 * x                                                   # Horizontal
# Trial points start on the section, but may move anywhere on the water.
trial = (x * seed_offsets).cast(Horizontal)                      # [paths] Horizontal
for _ in range(n_iterations):
    # The local ray map turns the miss at the receiver into a correction on the water.
    trial = trial - derivative(ripple_landing)(trial).solve(ripple_landing(trial) - target)   # [paths] Horizontal

section = x * np.linspace(-section_halfwidth, section_halfwidth, n_section)   # [samples] Horizontal
fan = x * np.linspace(-section_halfwidth, section_halfwidth, n_fan)           # [rays] Horizontal
surface, _, _ = ripple(section)                                  # [samples] Direction
fan_surface, _, _ = ripple(fan)                                  # [rays] Direction
path_surface, _, _ = ripple(trial)                               # [paths] Direction
figure = render.draw_paths(surface, fan_surface, ripple_landing(fan) - up * depth, path_surface,
                           target - up * depth, incident_height)
save_figure(figure, "specular_caustics_paths")
figure

In [ ]:
# checks
# The ray map folds over part of the water, and the three paths reach the target from three distinct
# surface points.
assert area.to_array().min() < 0 < area.to_array().max()
np.testing.assert_allclose(ripple_landing(trial).kernel, 0, atol=1e-6)
crossings = (mv.x & path_surface).to_array()                     # [paths]
assert crossings[0] < -0.2 and abs(crossings[1]) < 1e-9 and crossings[2] > 0.2

The three orange paths obey the same refraction law and reach the same target. At a fold two branches merge and the correction map becomes singular. [Zeltner, Georgiev and Jakob's *Specular Manifold Sampling*](https://rgl.epfl.ch/publications/Zeltner2020Specular) finds light paths with Newton solves of such specular constraints; its unbiased estimator also accounts for how often each solution is found, which this notebook leaves aside.